# 05 — v2 Results: HST-only Sources

This notebook explores the output of `bp3m-v2`, which extends the astrometric
catalogue to include HST-detected sources with no Gaia counterpart.

**Requires:** a completed `bp3m-v2` run (i.e. `BP3M_v2_results/` and `hst_xmatch/` must exist).

**Plots:** Gaia-matched vs HST-only populations · VPD · PM uncertainty comparison · CMD

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from bp3m.pipeline.explore_utils import (
    load_gaia_catalog, load_bp3m_results,
    gaia_pm_sigma, bp3m_pm_sigma, bp3m_full_cov, vpd_limits,
)

DATA = Path(OUTPUT_DIR)
field_dir = DATA / FIELD_NAME

# Load v1 BP3M results
bp3m_v1 = load_bp3m_results(field_dir / 'BP3M_results')
stars_v1 = bp3m_v1['stars']

# Load v2 BP3M results
try:
    bp3m_v2 = load_bp3m_results(field_dir / 'BP3M_v2_results')
    stars_v2 = bp3m_v2['stars']
    print(f'v2 stars: {len(stars_v2)}')
except FileNotFoundError:
    stars_v2 = None
    print('BP3M_v2_results not found — run bp3m-v2 first.')

# Load master HST cross-match catalog
master_csv = field_dir / 'hst_xmatch' / 'master_combined_v2.csv'
try:
    master = pd.read_csv(master_csv, low_memory=False)
    print(f'Master catalog: {len(master)} sources')
except FileNotFoundError:
    master = None
    print(f'master_combined_v2.csv not found at {master_csv}')

# Load Gaia catalog
_gaia_files = sorted((field_dir / 'Gaia').glob('*_gaia.csv'))
gaia = load_gaia_catalog(_gaia_files[0])
print(f'v1 stars: {len(stars_v1)}   Gaia: {len(gaia)}')

In [ ]:
# Separate Gaia-matched and HST-only sources in v2 results
# HST-only sources have negative Gaia_id (synthetic IDs assigned by bp3m)
if stars_v2 is not None:
    gaia_ids_v2 = stars_v2['Gaia_id'].values.astype(np.int64)
    is_gaia_matched = gaia_ids_v2 > 0
    is_hst_only     = gaia_ids_v2 < 0

    print(f'v2 Gaia-matched: {is_gaia_matched.sum()}')
    print(f'v2 HST-only:     {is_hst_only.sum()}')

    pmra_v2_gaia  = stars_v2.loc[is_gaia_matched, 'pmra_bp3m'].values
    pmdec_v2_gaia = stars_v2.loc[is_gaia_matched, 'pmdec_bp3m'].values
    pmra_v2_hst   = stars_v2.loc[is_hst_only, 'pmra_bp3m'].values
    pmdec_v2_hst  = stars_v2.loc[is_hst_only, 'pmdec_bp3m'].values
    gmag_v2_gaia  = stars_v2.loc[is_gaia_matched, 'gmag'].values
    n_hst_v2      = stars_v2['n_hst_used'].values

In [ ]:
# ── VPD: v2 Gaia-matched vs HST-only ─────────────────────────────────────────
if stars_v2 is not None:
    xlim, ylim = vpd_limits(
        np.concatenate([pmra_v2_gaia, pmra_v2_hst]),
        np.concatenate([pmdec_v2_gaia, pmdec_v2_hst]),
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

    axes[0].scatter(pmra_v2_gaia, pmdec_v2_gaia,
                    s=4, alpha=0.5, c='steelblue', rasterized=True,
                    label=f'Gaia-matched ({is_gaia_matched.sum()})')
    axes[0].set_title('v2 — Gaia-matched sources')

    axes[1].scatter(pmra_v2_hst, pmdec_v2_hst,
                    s=4, alpha=0.5, c='darkorange', rasterized=True,
                    label=f'HST-only ({is_hst_only.sum()})')
    axes[1].set_title('v2 — HST-only sources')

    for ax in axes:
        ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.3)
        ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.3)
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_xlabel(r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)')
        ax.legend(fontsize=9)
    axes[0].set_ylabel(r'$\mu_\delta$ (mas yr$^{-1}$)')

    fig.suptitle(f'{FIELD_NAME} — v2 Vector Point Diagram', fontsize=13)
    plt.tight_layout()
    plt.savefig(field_dir / 'plots_05_vpd_v2.png', dpi=150)
    plt.show()

In [ ]:
# ── PM uncertainty comparison: Gaia / v1 / v2 ────────────────────────────────
if stars_v2 is not None:
    sig_pm_gaia = gaia_pm_sigma(gaia)
    sig_pm_v1   = bp3m_pm_sigma(bp3m_v1)
    C_full_v2   = bp3m_full_cov(bp3m_v2)
    from bp3m.pipeline.catalog_utils import pm_uncertainty
    sig_pm_v2_all = pm_uncertainty(C_full_v2)

    sig_pm_v2_gaia = sig_pm_v2_all[is_gaia_matched]
    sig_pm_v2_hst  = sig_pm_v2_all[is_hst_only]
    gmag_gaia_all  = gaia['gmag'].values

    fig, ax = plt.subplots(figsize=(9, 6))

    ok_g = np.isfinite(gmag_gaia_all) & np.isfinite(sig_pm_gaia)
    ax.scatter(gmag_gaia_all[ok_g], sig_pm_gaia[ok_g],
               s=4, alpha=0.3, c='grey', rasterized=True, label='Gaia DR3')

    ok_v1 = np.isfinite(stars_v1['gmag'].values) & np.isfinite(sig_pm_v1)
    ax.scatter(stars_v1['gmag'].values[ok_v1], sig_pm_v1[ok_v1],
               s=4, alpha=0.4, c='steelblue', rasterized=True, label='BP3M v1')

    ok_v2g = np.isfinite(gmag_v2_gaia) & np.isfinite(sig_pm_v2_gaia)
    ax.scatter(gmag_v2_gaia[ok_v2g], sig_pm_v2_gaia[ok_v2g],
               s=4, alpha=0.5, c='darkorange', rasterized=True,
               label='BP3M v2 (Gaia-matched)')

    ok_v2h = np.isfinite(sig_pm_v2_hst)
    n_hst_v2_hst = n_hst_v2[is_hst_only]
    sc = ax.scatter(
        np.full(ok_v2h.sum(), np.nan),   # HST-only have no Gaia G — use n_hst as proxy
        sig_pm_v2_hst[ok_v2h],
        s=6, alpha=0.0)                  # invisible placeholder for colorbar

    # Plot HST-only by N_HST instead of G magnitude
    fig2, ax2 = plt.subplots(figsize=(7, 5))
    sc2 = ax2.scatter(n_hst_v2_hst[ok_v2h], sig_pm_v2_hst[ok_v2h],
                      s=6, alpha=0.5, c='darkorange', rasterized=True)
    ax2.set_xlabel('N HST detections')
    ax2.set_ylabel(r'$\sigma_\mu$ (mas yr$^{-1}$)')
    ax2.set_yscale('log')
    ax2.set_title(f'{FIELD_NAME} — v2 HST-only PM uncertainty vs N detections')
    plt.tight_layout()
    plt.savefig(field_dir / 'plots_05_pm_unc_hst_only.png', dpi=150)
    plt.show()

    ax.set_xlabel('Gaia G (mag)')
    ax.set_ylabel(r'$(\det C_\mu)^{1/4}$ (mas yr$^{-1}$)')
    ax.set_yscale('log')
    ax.legend(fontsize=9)
    ax.set_title(f'{FIELD_NAME} — PM uncertainty: Gaia / v1 / v2 Gaia-matched')
    fig.tight_layout()
    fig.savefig(field_dir / 'plots_05_pm_unc_comparison.png', dpi=150)
    plt.show()

In [ ]:
# ── CMD from master cross-match catalog ──────────────────────────────────────
# The master_combined_v2.csv contains per-filter normalised magnitudes.
# We look for two filter columns and build a colour-magnitude diagram.
if master is not None:
    # Find available filter magnitude columns
    mag_cols = [c for c in master.columns
                if c.startswith('mag_norm_wmean_') or
                   (c.startswith('mag_') and 'wmean' in c)]
    print(f'Available magnitude columns: {mag_cols}')

    if len(mag_cols) >= 2:
        # Use first two filters found (sorted so bluer filter is first)
        col1, col2 = sorted(mag_cols)[:2]
        filt1 = col1.split('_')[-1] if '_' in col1 else col1
        filt2 = col2.split('_')[-1] if '_' in col2 else col2

        ok = np.isfinite(master[col1].values) & np.isfinite(master[col2].values)
        color_hst = master.loc[ok, col1].values - master.loc[ok, col2].values
        mag_hst   = master.loc[ok, col2].values

        # Separate Gaia-recovered vs HST-only in master catalog
        has_gaia = master.loc[ok, 'gaia_source_id'].notna() \
                   if 'gaia_source_id' in master.columns else \
                   pd.Series(False, index=master.index[ok])

        fig, axes = plt.subplots(1, 2, figsize=(13, 6))

        # Left: Gaia CMD
        has_color = np.isfinite(gaia['bp_rp'].values) if 'bp_rp' in gaia.columns \
                    else np.zeros(len(gaia), bool)
        if has_color.any():
            axes[0].scatter(gaia.loc[has_color, 'bp_rp'],
                            gaia.loc[has_color, 'gmag'],
                            s=3, alpha=0.4, c='steelblue', rasterized=True)
        axes[0].invert_yaxis()
        axes[0].set_xlabel('BP − RP (mag)')
        axes[0].set_ylabel('Gaia G (mag)')
        axes[0].set_title('Gaia CMD')

        # Right: HST CMD from master catalog
        axes[1].scatter(color_hst[~has_gaia.values], mag_hst[~has_gaia.values],
                        s=3, alpha=0.4, c='darkorange', rasterized=True,
                        label='HST-only')
        axes[1].scatter(color_hst[has_gaia.values], mag_hst[has_gaia.values],
                        s=3, alpha=0.4, c='steelblue', rasterized=True,
                        label='Gaia-matched')
        axes[1].invert_yaxis()
        axes[1].set_xlabel(f'{filt1} − {filt2} (mag)')
        axes[1].set_ylabel(f'{filt2} (mag)')
        axes[1].set_title('HST CMD (master catalog)')
        axes[1].legend(fontsize=9)

        fig.suptitle(f'{FIELD_NAME} — Colour-magnitude diagrams', fontsize=13)
        plt.tight_layout()
        plt.savefig(field_dir / 'plots_05_cmd.png', dpi=150)
        plt.show()
    else:
        print(f'Need ≥2 filter magnitude columns for CMD. Found: {mag_cols}')
        print(f'Available master columns: {list(master.columns)}')
else:
    print('master_combined_v2.csv required for CMD.')

In [ ]:
# ── Sky distribution: v2 Gaia-matched vs HST-only ────────────────────────────
if stars_v2 is not None and 'ra' in stars_v2.columns and 'dec' in stars_v2.columns:
    ra_gaia  = stars_v2.loc[is_gaia_matched, 'ra'].values
    dec_gaia = stars_v2.loc[is_gaia_matched, 'dec'].values
    ra_hst   = stars_v2.loc[is_hst_only, 'ra'].values
    dec_hst  = stars_v2.loc[is_hst_only, 'dec'].values

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(ra_gaia, dec_gaia, s=3, alpha=0.4, c='steelblue',
               rasterized=True, label=f'Gaia-matched ({is_gaia_matched.sum()})')
    ax.scatter(ra_hst, dec_hst, s=3, alpha=0.4, c='darkorange',
               rasterized=True, label=f'HST-only ({is_hst_only.sum()})')

    all_ra  = np.concatenate([ra_gaia, ra_hst])
    all_dec = np.concatenate([dec_gaia, dec_hst])
    pad_ra  = (all_ra.max()  - all_ra.min())  * 0.05
    pad_dec = (all_dec.max() - all_dec.min()) * 0.05
    ax.set_xlim(all_ra.max() + pad_ra, all_ra.min() - pad_ra)   # RA right-to-left
    ax.set_ylim(all_dec.min() - pad_dec, all_dec.max() + pad_dec)
    ax.set_xlabel('R.A. (deg)')
    ax.set_ylabel('Dec. (deg)')
    ax.set_title(f'{FIELD_NAME} — v2 sky distribution')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(field_dir / 'plots_05_sky_v2.png', dpi=150)
    plt.show()